In [ ]:
#Notebook description

#This notebook is being used to evaluate the techinical market conditions of a single asset and assess
#the appropriate strategy to take in order to maximize returns.

In [ ]:
#take all compuation functions and put them in a separate file

#simplify the date x axis on the percent drawdown chart

#default the zoom range to a comfortable range, and create a dropdown to select the time range for Volatility section


#Remove the VIX charting, its redudnant now that we have the volatility models

In [ ]:
# Block: load core libraries and instantiate helpers and plotters
# Load libraries
import logging
import warnings
import json
import os
import numpy as np
from scipy.stats import skew, kurtosis
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from Quantapp.visualization import Plotter, LineChartPlotter, CandleStickPlotter, BarChartPlotter
from Quantapp.analytics import Rolling, Helper, Models
from Quantapp.data import MacroDataClient
warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
logger.disabled = True
logger.propagate = False
rolling = Rolling()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()
lineChartPlotter = LineChartPlotter()
candleStickPlotter = CandleStickPlotter()
barChartPlotter = BarChartPlotter()


In [ ]:
# Block: define timeframe parameters and risk-free rate

#PARAMETERS (Adjust parmeters here)
# Define parameters for the analysis (Adjust these as needed)
interval = '1d'
period     = '20y'
risk_free_rate = 0.02 / 252  # Annualized risk-free rate divided by trading days
#should be a string or None
ticker_str = 'UNH'#ticker to analyze
benchmark_tickers = ['XLV']
length_of_plots = 20 #number of years of data to plot (after computing the period, this will be adjusted to the closest available data)
trading_strategy = 'position'  # Options: 'position', 'swing', or 'structural', # Determines the trading strategy to use for time frames


In [ ]:
# Block: organize structural, position, swing timeframe lists

# Load data
# Define timeframe lists for different strategies
structural_time_frames = [200, 500, 1000]  # For structural analysis (long-term)
position_time_frames = [21, 50, 200]  # For position trading (longer term)
swing_time_frames = [3, 9, 21]        # For swing trading (shorter term)

# Populate the time frame variables based on selected strategy
if trading_strategy == "position":
    time_frame_short = position_time_frames[0]
    time_frame_mid = position_time_frames[1]
    time_frame_long = position_time_frames[2]
elif trading_strategy == "structural":
    time_frame_short = structural_time_frames[0]
    time_frame_mid = structural_time_frames[1]
    time_frame_long = structural_time_frames[2]
else:  # swing trading
    time_frame_short = swing_time_frames[0]
    time_frame_mid = swing_time_frames[1]
    time_frame_long = swing_time_frames[2]

time_frame_map = {
    'short': time_frame_short,
    'mid': time_frame_mid,
    'long': time_frame_long
}

return_frequencies = ('monthly', 'weekly', 'daily')

def normalize_benchmark_tickers(benchmark_tickers, asset_ticker):
    normalized = []
    seen = set()
    asset_ticker = asset_ticker.upper()
    for symbol in benchmark_tickers:
        if symbol is None:
            continue
        symbol = str(symbol).strip().upper()
        if not symbol or symbol == asset_ticker or symbol in seen:
            continue
        normalized.append(symbol)
        seen.add(symbol)
    return normalized

def load_benchmark_data(benchmark_tickers, period, interval, helper):
    benchmark_frames = {}
    skipped = []
    for symbol in benchmark_tickers:
        frame = yf.Ticker(symbol).history(period=period, interval=interval)
        if frame.empty:
            skipped.append(symbol)
            continue
        benchmark_frames[symbol] = helper.simplify_datetime_index(frame)
    return benchmark_frames, skipped

def align_series_to_common_index(ticker_frame, vix_frame, benchmark_frames):
    analysis_index = ticker_frame.index.intersection(vix_frame.index)
    for benchmark_frame in benchmark_frames.values():
        analysis_index = analysis_index.intersection(benchmark_frame.index)
    analysis_index = analysis_index.sort_values()
    aligned_benchmarks = {symbol: frame.loc[analysis_index] for symbol, frame in benchmark_frames.items()}
    return analysis_index, ticker_frame.loc[analysis_index], vix_frame.loc[analysis_index], aligned_benchmarks

# Download and normalize asset-level data
ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = yf.Ticker('^VIX').history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
vix = helper.simplify_datetime_index(vix)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: rolling.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: rolling.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: rolling.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']


In [ ]:
# Block: display regime-change candlestick with Bollinger bands

#REGIME CHANGES: Candlestick with RSI and Bollinger Bands
candleStickPlotter.create_candlestick_fig(ticker, period='2y', bollinger_window=50, title="Candlestick with 50-Period Bollinger Bands")

In [ ]:
# Block: plot percentage drop from historical peaks

#REGIME CHANGES: Percentage Drop from Highest Peak
n = int(252 / 2)
qp.plot_percentage_drop(ticker, n=n, title=f'{ticker_str} Percentage Drop from Highest Peak')

In [ ]:
# Block: compute rolling Sharpe windows, momentum histograms, and volatility

# --- 1. Load your price series ---
# Replace this with your actual data loading code, make sure `ticker` DataFrame has 'Close' column
# For example:
# ticker = yf.download('SPY', start='2010-01-01', end='2025-07-12')
price_series = pd.Series(ticker['Close'].asfreq('D')).dropna()

# Define your window sizes as integers
window_sizes = list(range(3, 201))

# --- 2. Compute rolling Sharpe ratios and determine optimal window ---

def compute_optimal_momentum_window(price_series, windows):
    returns = price_series.pct_change()
    sharpe_df = pd.DataFrame(index=price_series.index)

    for w in windows:
        mean_return = returns.rolling(window=w).mean()
        std_return = returns.rolling(window=w).std()
        sharpe = mean_return / (std_return + 1e-8)  # prevent divide by zero
        sharpe_df[w] = sharpe

    # Drop rows where all Sharpe ratios are NaN (start of series)
    sharpe_df = sharpe_df.dropna(how='all')

    # Find the window with max Sharpe ratio at each date
    sharpe_df['Optimal_Window'] = sharpe_df.idxmax(axis=1).astype(float)
    return sharpe_df

sharpe_table = compute_optimal_momentum_window(price_series, window_sizes)

# --- 3. Plot optimal momentum window over time ---

fig_optimal = go.Figure()
fig_optimal.add_trace(go.Scatter(
    x=sharpe_table.index,
    y=sharpe_table['Optimal_Window'],
    mode='lines',
    name='Optimal Window Size'
))
fig_optimal.update_layout(
    title="Rolling Optimal Momentum Window",
    xaxis_title="Date",
    yaxis_title="Window Size (Days)",
    template="plotly_white",
    showlegend=False
)
fig_optimal.show()



# Extract optimal window series, drop NaNs if any
optimal_windows = sharpe_table['Optimal_Window'].dropna()

# Convert to integers (window sizes are floats from idxmax)
optimal_windows_int = optimal_windows.astype(int)

# Plot histogram of optimal window sizes
fig_dist = px.histogram(
    optimal_windows_int,
    nbins=len(window_sizes),
    labels={'value': 'Optimal Window Size (Days)', 'count': 'Frequency'},
    title='Distribution of Optimal Sharpe Momentum Windows Over Time'
)

# plot vertical lines at 21, 50, and 200 days
fig_dist.add_vline(x=7, line_color='orange', line_dash='dash',
                    annotation_text="7 Days", annotation_position="top left")

fig_dist.add_vline(x=21, line_color='red', line_dash='dash',
                    annotation_text="21 Days", annotation_position="top left")
fig_dist.add_vline(x=50, line_color='blue', line_dash='dash',
                    annotation_text="50 Days", annotation_position="top left")
fig_dist.add_vline(x=200, line_color='green', line_dash='dash',
                    annotation_text="200 Days", annotation_position="top left")

fig_dist.update_layout(bargap=0.1)
fig_dist.show()

# --- 4. Calculate mean and median Sharpe ratios across all dates ---

# Exclude the 'Optimal_Window' column for stats
sharpe_only = sharpe_table.drop(columns='Optimal_Window')

mean_sharpe = sharpe_only.mean()
median_sharpe = sharpe_only.median()

fig_mean_median = go.Figure()
fig_mean_median.add_trace(go.Scatter(
    x=mean_sharpe.index,
    y=mean_sharpe.values,
    mode='lines+markers',
    name='Mean Sharpe Ratio'
))
fig_mean_median.add_trace(go.Scatter(
    x=median_sharpe.index,
    y=median_sharpe.values,
    mode='lines+markers',
    name='Median Sharpe Ratio'
))
fig_mean_median.update_layout(
    title="Mean and Median Sharpe Ratios Across All Dates by Momentum Window",
    xaxis_title="Momentum Window Size (Days)",
    yaxis_title="Sharpe Ratio",
    template="plotly_white"
)

# Plot vertical lines at 21, 50, and 200 days
fig_mean_median.add_vline(x=7, line_color='orange', line_dash='dash',
                    annotation_text="7 Days", annotation_position="top left")
fig_mean_median.add_vline(x=21, line_color='red', line_dash='dash',
                    annotation_text="21 Days", annotation_position="top left")
fig_mean_median.add_vline(x=50, line_color='blue', line_dash='dash',
                    annotation_text="50 Days", annotation_position="top left")
fig_mean_median.add_vline(x=200, line_color='green', line_dash='dash',
                    annotation_text="200 Days", annotation_position="top left")

fig_mean_median.show()

# --- 5. Calculate rolling volatility for each window ---

returns = price_series.pct_change()
volatility_df = pd.DataFrame(index=price_series.index)

for w in window_sizes:
    volatility_df[w] = returns.rolling(window=w).std()

volatility_df = volatility_df.reindex(sorted(volatility_df.columns), axis=1)

mean_volatility = volatility_df.mean()
median_volatility = volatility_df.median()

# --- 6. Plot mean and median volatility ---

fig_vol = go.Figure()
fig_vol.add_trace(go.Scatter(
    x=mean_volatility.index,
    y=mean_volatility.values,
    mode='lines+markers',
    name='Mean Volatility'
))
fig_vol.add_trace(go.Scatter(
    x=median_volatility.index,
    y=median_volatility.values,
    mode='lines+markers',
    name='Median Volatility'
))
fig_vol.update_layout(
    title="Mean and Median Volatility Across All Dates by Momentum Window",
    xaxis_title="Momentum Window Size (Days)",
    yaxis_title="Volatility (Rolling Std of Returns)",
    template="plotly_white"
)
#plot vertical line at 21, 50, and 200 days
fig_vol.add_vline(x=7, line_color='orange', line_dash='dash',
                    annotation_text="7 Days", annotation_position="top left")
fig_vol.add_vline(x=21, line_color='red', line_dash='dash',
                    annotation_text="21 Days", annotation_position="top left")
fig_vol.add_vline(x=50, line_color='blue', line_dash='dash',
                    annotation_text="50 Days", annotation_position="top left")
fig_vol.add_vline(x=200, line_color='green', line_dash='dash',
                    annotation_text="200 Days", annotation_position="top left")

fig_vol.show()

#print("Mean volatility at selected windows:")
#print(mean_volatility.loc[[5, 21, 50, 100, 150, 200]])
#print("\nMedian volatility at selected windows:")
#print(median_volatility.loc[[5, 21, 50, 100, 150, 200]])

# --- 7. 3D Surface plot of Sharpe ratios over last 10 years with vertical planes ---

# Filter to last 10 years of data
last_date = sharpe_only.index[-1]
ten_years_ago = last_date - pd.Timedelta(days=365*10)
sharpe_last_10y = sharpe_only.loc[ten_years_ago:last_date]

# Create integer y-axis for dates (since 3D plots need numeric axes)
date_labels = sharpe_last_10y.index.strftime('%Y-%m-%d')
y_vals = np.arange(len(sharpe_last_10y))

def create_vertical_plane(x_val, y_vals, z_min, z_max, opacity=0.3, color='red'):
    y_grid, z_grid = np.meshgrid(y_vals, np.linspace(z_min, z_max, 2))
    x_grid = np.full_like(y_grid, x_val)
    return go.Surface(
        x=x_grid,
        y=y_grid,
        z=z_grid,
        showscale=False,
        opacity=opacity,
        colorscale=[[0, color], [1, color]],
        hoverinfo='skip'
    )

surface_trace = go.Surface(
    z=sharpe_last_10y.values,
    x=sharpe_last_10y.columns,
    y=y_vals,
    colorscale='Viridis',
    colorbar=dict(title='Sharpe Ratio'),
    name='Sharpe Surface'
)

plane_21 = create_vertical_plane(21, y_vals, np.nanmin(sharpe_last_10y.values), np.nanmax(sharpe_last_10y.values), opacity=0.3, color='red')
plane_50 = create_vertical_plane(50, y_vals, np.nanmin(sharpe_last_10y.values), np.nanmax(sharpe_last_10y.values), opacity=0.3, color='blue')
plane_200 = create_vertical_plane(200, y_vals, np.nanmin(sharpe_last_10y.values), np.nanmax(sharpe_last_10y.values), opacity=0.3, color='green')

fig_3d = go.Figure(data=[surface_trace, plane_21, plane_50, plane_200])

fig_3d.update_layout(
    title='3D Surface of Sharpe Ratios by Window and Date (Last 10 Years) with Highlighted Windows',
    scene=dict(
        xaxis_title='Momentum Window Size (Days)',
        yaxis_title='Date',
        yaxis=dict(
            tickmode='array',
            tickvals=np.arange(0, len(date_labels), step=365),  # roughly yearly ticks
            ticktext=date_labels[::365]
        ),
        zaxis_title='Sharpe Ratio'
    ),
    height=900,
    template="plotly_white"
)

fig_3d.show()


In [ ]:
# Block: build VIX Fix series and overlay standard deviation bands

#Volatility: VIX FIX 

ticker_vix_fix = rolling.vix_fix(ticker)
benchmark_vix_fix = {symbol: rolling.vix_fix(frame) for symbol, frame in benchmark_data.items()}

qp.plot_series_with_stdev_bands(
    ticker_vix_fix,
    title='VIX Fix with Mean and Standard Deviations',
    stdev_values=[-0.5, 0.5, 1.5, 3]
)


In [ ]:
# Block: visualize monthly, weekly, and daily seasonality patterns

#Seasonality: Monthly, Weekly, and Daily Returns
#SEASONALITY: Monthly Seasonality
fig_ticker_Seasonality_Monthly = barChartPlotter.create_seasonality_fig(ticker_monthly_returns, f'Monthly Seasonality: {ticker_str}', 'monthly')
fig_ticker_Seasonality_Monthly.show()

#SEASONALITY: Weekly Seasonality
fig_ticker_Seasonality_weekly = barChartPlotter.create_seasonality_fig(ticker_weekly_returns, f'Weekly Seasonality: {ticker_str}', 'weekly')
fig_ticker_Seasonality_weekly.show()

#SEASONALITY: Daily Seasonality
fig_ticker_Seasonality_daily = barChartPlotter.create_seasonality_fig(ticker_daily_returns, f'Daily Seasonality: {ticker_str}', 'daily')
fig_ticker_Seasonality_daily.show()

In [ ]:
# Block: compute log-price MACD plus histogram z-score helpers
'''
def calculate_macd_log(price, short_window=12, long_window=26, signal_window=9):
    # Calculate log of close prices
    log_close = np.log(price['Close'])
    
    # Compute EMAs on log prices
    ema_short = log_close.ewm(span=short_window, adjust=True).mean()
    ema_long = log_close.ewm(span=long_window, adjust=True).mean()
    
    macd_line = ema_short - ema_long
    signal_line = macd_line.ewm(span=signal_window, adjust=True).mean()
    histogram = macd_line - signal_line
    
    return macd_line, signal_line, histogram

def calculate_zscore(series):
    return (series - series.mean()) / series.std()

def plot_macd_with_histogram_zscore(macd_line, signal_line, histogram, price_data, title="MACD (log prices) and Histogram Z-Score"):
    zscore_series = calculate_zscore(histogram.dropna())

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.4, 0.6], vertical_spacing=0.05,
                        subplot_titles=("MACD Line and Signal Line", "Histogram Z-Score"))

    # MACD line
    fig.add_trace(go.Scatter(
        x=macd_line.index, y=macd_line,
        mode='lines', name='MACD Line',
        line=dict(color='purple')
    ), row=1, col=1)

    # Signal line
    fig.add_trace(go.Scatter(
        x=signal_line.index, y=signal_line,
        mode='lines', name='Signal Line',
        line=dict(color='orange')
    ), row=1, col=1)

    # Histogram as bar chart
    fig.add_trace(go.Bar(
        x=histogram.index, y=histogram,
        name='Histogram',
        marker_color='gray'
    ), row=1, col=1)

    # Histogram z-score plot (bottom)
    fig.add_trace(go.Scatter(
        x=zscore_series.index, y=zscore_series,
        mode='lines', name='Histogram Z-Score',
        line=dict(color='blue')
    ), row=2, col=1)

    # Add horizontal lines and shaded regions for z-score
    levels = [0.5, 1, 1.5]
    for level in levels:
        fig.add_hline(y=level, line_color='red', line_dash='dash',
                      annotation_text=f"+{level} SD", annotation_position="top right", row=2, col=1)
        fig.add_hline(y=-level, line_color='green', line_dash='dash',
                      annotation_text=f"-{level} SD", annotation_position="bottom right", row=2, col=1)

    fig.add_hline(y=0, line_color='black', line_dash='dot', row=2, col=1)

    fig.add_shape(
        type="rect",
        xref="x", yref="y2",
        x0=zscore_series.index[0], x1=zscore_series.index[-1],
        y0=1, y1=1.5,
        fillcolor="rgba(255, 0, 0, 0.2)", line_width=0, layer="below",
        row=2, col=1
    )
    fig.add_shape(
        type="rect",
        xref="x", yref="y2",
        x0=zscore_series.index[0], x1=zscore_series.index[-1],
        y0=-1.5, y1=-1,
        fillcolor="rgba(0, 128, 0, 0.2)", line_width=0, layer="below",
        row=2, col=1
    )

    fig.update_layout(
        height=900,
        title=title,
        template="plotly_white",
        yaxis_title="MACD",
        yaxis2_title="Histogram Z-Score"
    )

    fig.show()

# Usage:
macd_12, signal_12, hist_12 = calculate_macd_log(ticker, 12, 26)
macd_21, signal_21, hist_21 = calculate_macd_log(ticker, 21, 50)
macd_50, signal_50, hist_50 = calculate_macd_log(ticker, 50, 200)
macd_200, signal_200, hist_200 = calculate_macd_log(ticker, 200, 400)

plot_macd_with_histogram_zscore(macd_12, signal_12, hist_12, ticker, title="Log-Price MACD and Histogram Z-Score (12, 26)")
plot_macd_with_histogram_zscore(macd_21, signal_21, hist_21, ticker, title="Log-Price MACD and Histogram Z-Score (21, 50)")
plot_macd_with_histogram_zscore(macd_50, signal_50, hist_50, ticker, title="Log-Price MACD and Histogram Z-Score (50, 200)")
plot_macd_with_histogram_zscore(macd_200, signal_200, hist_200, ticker, title="Log-Price MACD and Histogram Z-Score (200, 400)")

'''


In [ ]:
# Block: render interactive momentum z-score comparisons

#Momentum Z-Score with Interactive Window Selection

def calculate_average_return(price, window):
    daily_returns = price['Close'].pct_change()
    avg_return = daily_returns.rolling(window=window).mean() * 100
    return avg_return

def calculate_momentum_zscore(price, short_window, long_window):
    avg_short = calculate_average_return(price, short_window)
    avg_long = calculate_average_return(price, long_window)
    momentum_diff = avg_short - avg_long
    zscore_series = (momentum_diff - momentum_diff.mean()) / momentum_diff.std()
    return zscore_series

# Define window pairs
window_pairs = {
    "21 vs 50": (21, 50),
    "50 vs 200": (50, 200),
    "200 vs 400": (200, 400)
}

# Calculate z-score for each window pair
zscore_data = {}
for label, (short_w, long_w) in window_pairs.items():
    zscore_data[label] = calculate_momentum_zscore(ticker, short_w, long_w)

# Create figure
fig = go.Figure()

for label, series in zscore_data.items():
    fig.add_trace(go.Scatter(
        x=series.index, y=series,
        mode='lines',
        name=label,
        visible=(label == "21 vs 50")
    ))

# Dropdown for moving window pair
buttons_window = []
for i, label in enumerate(zscore_data.keys()):
    visible = [False] * len(zscore_data)
    visible[i] = True
    buttons_window.append(dict(
        label=label,
        method="update",
        args=[{"visible": visible},
              {"title": f"Momentum Z-Score {label}"}]
    ))

# Dropdown for time range zoom
buttons_time = []
time_ranges = {
    "1 Year": 252,
    "3 Years": 252*3,
    "5 Years": 252*5,
    "10 Years": 252*10
}

for label, days in time_ranges.items():
    buttons_time.append(dict(
        label=label,
        method="relayout",
        args=[{"xaxis.range": [ticker.index[-days], ticker.index[-1]]}]
    ))

# Set default zoom to 3 years
default_days = 252*3
x_range_default = [ticker.index[-default_days], ticker.index[-1]]

fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            showactive=True,
            buttons=buttons_window,
            x=0.1, y=1.15,
            xanchor="left",
            direction="down"
        ),
        dict(
            type="dropdown",
            showactive=True,
            buttons=buttons_time,
            x=0.3, y=1.15,
            xanchor="left",
            direction="down"
        )
    ],
    height=600,
    title="Momentum Z-Score 21 vs 50",
    template="plotly_white",
    yaxis_title="Z-Score",
    xaxis=dict(range=x_range_default)
)

# Horizontal lines
for level in [0.5, 1, 1.5]:
    fig.add_hline(y=level, line_color='red', line_dash='dash')
    fig.add_hline(y=-level, line_color='green', line_dash='dash')
fig.add_hline(y=0, line_color='black', line_dash='dot')

# Shaded regions
fig.add_shape(type="rect", xref="paper", x0=0, x1=1, yref="y", y0=-1.5, y1=-1,
              fillcolor="rgba(0, 255, 0, 0.2)", line_width=0, layer="below")
fig.add_shape(type="rect", xref="paper", x0=0, x1=1, yref="y", y0=1, y1=1.5,
              fillcolor="rgba(255, 0, 0, 0.2)", line_width=0, layer="below")

fig.show()


In [ ]:
# Block: compute Sharpe/Sortino ratios and spreads

#Calculations

asset_close = ticker['Close']

asset_sharpe_map = {
    term: rolling.risk_adjusted_returns(asset_close, ratio_type='sharpe', windows=[window])[f'sharpe_ratio_{window}']
    for term, window in time_frame_map.items()
}
asset_sortino_map = {
    term: rolling.risk_adjusted_returns(asset_close, ratio_type='sortino', windows=[window])[f'sortino_ratio_{window}']
    for term, window in time_frame_map.items()
}

sharpe_ticker_short = asset_sharpe_map['short']
sharpe_ticker_mid = asset_sharpe_map['mid']
sharpe_ticker_long = asset_sharpe_map['long']

sortino_ticker_short = asset_sortino_map['short']
sortino_ticker_mid = asset_sortino_map['mid']
sortino_ticker_long = asset_sortino_map['long']

# calculate the spread between sharpe and sortino ratios of the ticker
sharpe_sortino_spread_short = sharpe_ticker_short - sortino_ticker_short
sharpe_sortino_spread_mid = sharpe_ticker_mid - sortino_ticker_mid
sharpe_sortino_spread_long = sharpe_ticker_long - sortino_ticker_long

def compute_benchmark_metrics(benchmark_data, asset_close, asset_sharpe_map, rolling, time_frame_map):
    metrics = {}
    for symbol, benchmark_frame in benchmark_data.items():
        benchmark_close = benchmark_frame['Close']
        metrics[symbol] = {}
        for term, window in time_frame_map.items():
            benchmark_sharpe = rolling.risk_adjusted_returns(
                benchmark_close,
                ratio_type='sharpe',
                windows=[window]
            )[f'sharpe_ratio_{window}']
            relative_spread = benchmark_close.pct_change(window) - asset_close.pct_change(window)
            metrics[symbol][term] = {
                'spread': relative_spread,
                'sharpe_ratio': benchmark_sharpe,
                'sharpe_spread': benchmark_sharpe - asset_sharpe_map[term]
            }
    return metrics

benchmark_metrics = compute_benchmark_metrics(benchmark_data, asset_close, asset_sharpe_map, rolling, time_frame_map)
benchmark_order = list(benchmark_metrics.keys())
default_benchmark = benchmark_order[0] if benchmark_order else None
spread_plot_data = {
    term: {symbol: benchmark_metrics[symbol][term]['sharpe_spread'] for symbol in benchmark_order}
    for term in time_frame_map
}


In [ ]:
# Block: plot Sharpe & Sortino efficiency with dropdown controls

#Sharpe and Sortino Efficiency 

def plot_sharpe_sortino(term_config_map, ticker_str, default_label=None):
    if not term_config_map:
        raise ValueError("term_config_map is empty.")

    term_labels = list(term_config_map.keys())
    default_label = default_label if default_label in term_config_map else term_labels[0]

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            'Sharpe and Sortino Ratios',
            'Sharpe-Sortino Spread (z-score)'
        )
    )

    term_trace_map = {}

    for term_label in term_labels:
        cfg = term_config_map[term_label]
        sharpe = cfg['sharpe']
        sortino = cfg['sortino']
        time_frame = cfg['time_frame']

        mean_sharpe = sharpe.mean()
        mean_sortino = sortino.mean()

        spread = (sharpe - sortino).dropna()
        spread_mean = spread.mean()
        spread_std = spread.std(ddof=0) or 1.0
        z_spread = (spread - spread_mean) / spread_std
        z_index = z_spread.index

        visible = term_label == default_label
        trace_indices = []

        fig.add_trace(
            go.Scatter(
                x=sharpe.index,
                y=sharpe,
                mode='lines',
                name=f'Sharpe Ratio ({time_frame}-day)',
                line=dict(color='blue'),
                visible=visible,
                legendgroup='Sharpe'
            ),
            row=1,
            col=1
        )
        trace_indices.append(len(fig.data) - 1)

        fig.add_trace(
            go.Scatter(
                x=sortino.index,
                y=sortino,
                mode='lines',
                name=f'Sortino Ratio ({time_frame}-day)',
                line=dict(color='orange'),
                visible=visible,
                legendgroup='Sortino'
            ),
            row=1,
            col=1
        )
        trace_indices.append(len(fig.data) - 1)

        fig.add_trace(
            go.Scatter(
                x=sharpe.index,
                y=np.full(len(sharpe.index), mean_sharpe),
                mode='lines',
                name=f'Mean Sharpe ({mean_sharpe:.3f})',
                line=dict(color='blue', dash='dash'),
                opacity=0.7,
                visible=visible,
                legendgroup='Sharpe Mean',
                showlegend=False
            ),
            row=1,
            col=1
        )
        trace_indices.append(len(fig.data) - 1)

        fig.add_trace(
            go.Scatter(
                x=sortino.index,
                y=np.full(len(sortino.index), mean_sortino),
                mode='lines',
                name=f'Mean Sortino ({mean_sortino:.3f})',
                line=dict(color='orange', dash='dash'),
                opacity=0.7,
                visible=visible,
                legendgroup='Sortino Mean',
                showlegend=False
            ),
            row=1,
            col=1
        )
        trace_indices.append(len(fig.data) - 1)

        fig.add_trace(
            go.Scatter(
                x=z_index,
                y=z_spread,
                mode='lines',
                name=f'Sharpe-Sortino Spread ({time_frame}-day)',
                line=dict(color='green'),
                visible=visible,
                legendgroup='Spread'
            ),
            row=2,
            col=1
        )
        trace_indices.append(len(fig.data) - 1)

        for level, color in [(0, 'black'), (1, 'red'), (-1, 'red'), (2, 'purple'), (-2, 'purple')]:
            fig.add_trace(
                go.Scatter(
                    x=z_index,
                    y=np.full(len(z_index), level),
                    mode='lines',
                    line=dict(color=color, dash='dot'),
                    hoverinfo='skip',
                    showlegend=False,
                    visible=visible
                ),
                row=2,
                col=1
            )
            trace_indices.append(len(fig.data) - 1)

        fig.add_trace(
            go.Scatter(
                x=[z_index[-1]],
                y=[z_spread.mean()],
                mode='markers+text',
                text=[f'Mean z: {z_spread.mean():.2f}'],
                textposition='middle right',
                marker=dict(color='purple', size=6),
                showlegend=False,
                visible=visible
            ),
            row=2,
            col=1
        )
        trace_indices.append(len(fig.data) - 1)

        term_trace_map[term_label] = trace_indices

    buttons = []
    total_traces = len(fig.data)

    for term_label in term_labels:
        visibility = [False] * total_traces
        for idx in term_trace_map[term_label]:
            visibility[idx] = True

        buttons.append(
            dict(
                label=term_label,
                method='update',
                args=[
                    {'visible': visibility},
                    {'title': {'text': f'Sharpe & Sortino Analysis for {ticker_str} ({term_label})'}}
                ]
            )
        )

    fig.update_layout(
        template='plotly_white',
        height=900,
        legend=dict(x=0.01, y=0.99),
        xaxis2_title='Date',
        yaxis_title='Ratio Value',
        yaxis2_title='Spread (z-score)',
        title=dict(text=f'Sharpe & Sortino Analysis for {ticker_str} ({default_label})', x=0.5),
        updatemenus=[
            dict(
                buttons=buttons,
                direction='down',
                x=0.01,
                y=1.12,
                xanchor='left',
                yanchor='top',
                showactive=True,
                active=term_labels.index(default_label)
            )
        ]
    )

    fig.show()

term_config_map = {
    f'{time_frame_short}-day': {
        'sharpe': sharpe_ticker_short,
        'sortino': sortino_ticker_short,
        'time_frame': time_frame_short
    },
    f'{time_frame_mid}-day': {
        'sharpe': sharpe_ticker_mid,
        'sortino': sortino_ticker_mid,
        'time_frame': time_frame_mid
    },
    f'{time_frame_long}-day': {
        'sharpe': sharpe_ticker_long,
        'sortino': sortino_ticker_long,
        'time_frame': time_frame_long
    }
}

term = f'{time_frame_mid}-day' if f'{time_frame_mid}-day' in term_config_map else list(term_config_map.keys())[0]

plot_sharpe_sortino(term_config_map, ticker_str, default_label=term)

# ...existing code...

In [ ]:
# Block: combine risk-adjusted return and benchmark plots

#Risk-Adjusted Returns and Relative Returns Plots

def calculate_zscore(series):
    std = series.std()
    if std == 0 or pd.isna(std):
        return pd.Series(index=series.index, data=np.nan)
    return (series - series.mean()) / std

def add_std_reference_lines(fig, row, x_ref, levels=(1, 2, 3), visible=True):
    for level in levels:
        for value in (level, -level):
            fig.add_trace(
                go.Scatter(
                    x=x_ref,
                    y=[value] * len(x_ref),
                    mode='lines',
                    line=dict(color='rgba(220, 220, 220, 0.55)', dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    visible=visible
                ),
                row=row,
                col=1
            )

def add_horizontal_zone(fig, row, y0, y1, fillcolor, opacity=0.45):
    fig.add_hrect(
        y0=y0,
        y1=y1,
        fillcolor=fillcolor,
        opacity=opacity,
        line_color='green',
        line_width=1,
        layer='below',
        row=row,
        col=1
    )

def add_horizontal_zone_trace(fig, row, x_ref, y0, y1, fillcolor, visible=True):
    if x_ref is None or len(x_ref) == 0:
        return
    x0 = x_ref[0]
    x1 = x_ref[-1]
    fig.add_trace(
        go.Scatter(
            x=[x0, x1, x1, x0, x0],
            y=[y0, y0, y1, y1, y0],
            mode='lines',
            line=dict(width=0),
            fill='toself',
            fillcolor=fillcolor,
            hoverinfo='skip',
            showlegend=False,
            visible=visible
        ),
        row=row,
        col=1
    )

def build_time_range_buttons(global_start, global_end, axis_count=1):
    def make_range(years=None):
        if years is None:
            start = global_start
        else:
            start = max(global_start, global_end - pd.DateOffset(years=years))
        layout = {}
        for axis_idx in range(1, axis_count + 1):
            axis_name = 'xaxis' if axis_idx == 1 else f'xaxis{axis_idx}'
            layout[f'{axis_name}.range'] = [start, global_end]
        return layout

    return [
        dict(label='10 Years', method='relayout', args=[make_range(10)]),
        dict(label='5 Years', method='relayout', args=[make_range(5)]),
        dict(label='3 Years', method='relayout', args=[make_range(3)]),
        dict(label='1 Year', method='relayout', args=[make_range(1)]),
    ]

def make_multi_benchmark_summary_figure(spread_plot_data, time_frame_map, default_years, ticker_str):
    fig = make_subplots(rows=1, cols=1)
    term_order = ['short', 'mid', 'long']
    default_term = term_order[0]
    benchmark_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    term_ranges = {}
    term_trace_bounds = {}

    for term in term_order:
        term_series_map = spread_plot_data.get(term, {})
        visible = term == default_term
        non_empty_series = []
        term_zscore_map = {}
        x_ref = pd.Index([])
        for idx, symbol in enumerate(benchmark_order):
            series = term_series_map.get(symbol, pd.Series(dtype=float)).dropna()
            zscore_series = calculate_zscore(series).dropna() if not series.empty else pd.Series(dtype=float)
            term_zscore_map[symbol] = zscore_series
            if not zscore_series.empty:
                non_empty_series.append(zscore_series)
                if len(zscore_series.index) > len(x_ref):
                    x_ref = zscore_series.index
        term_trace_start = len(fig.data)
        add_horizontal_zone_trace(fig, 1, x_ref, -3, -2, 'rgba(180, 0, 0, 0.40)', visible=visible)
        add_horizontal_zone_trace(fig, 1, x_ref, 2, 3, 'rgba(0, 128, 0, 0.55)', visible=visible)
        for idx, symbol in enumerate(benchmark_order):
            zscore_series = term_zscore_map[symbol]
            fig.add_trace(
                go.Scatter(
                    x=zscore_series.index,
                    y=zscore_series,
                    mode='lines',
                    name=symbol,
                    legendgroup=symbol,
                    showlegend=visible,
                    line=dict(color=benchmark_palette[idx % len(benchmark_palette)]),
                    visible=visible
                ),
                row=1,
                col=1
            )
        add_std_reference_lines(fig, 1, x_ref, visible=visible)
        term_trace_bounds[term] = (term_trace_start, len(fig.data))
        if non_empty_series:
            max_index = max(series.index.max() for series in non_empty_series)
            min_index = min(series.index.min() for series in non_empty_series)
            term_ranges[term] = [max(min_index, max_index - pd.DateOffset(years=3)), max_index]
        else:
            term_ranges[term] = None

    timeframe_buttons = []
    total_traces = len(fig.data)
    for term in term_order:
        visibility = [False] * total_traces
        start, end = term_trace_bounds[term]
        for trace_idx in range(start, end):
            visibility[trace_idx] = True
        layout_updates = {
            'title': f"{ticker_str} {term.title()} Sharpe Spread Z-Scores vs Benchmarks ({time_frame_map[term]}-Day)",
            'yaxis': {'title': 'Sharpe Spread Z-Score'}
        }
        if term_ranges.get(term) is not None:
            layout_updates['xaxis'] = {'range': term_ranges[term]}
        timeframe_buttons.append(
            dict(
                label=f"{term.title()} ({time_frame_map[term]})",
                method='update',
                args=[{'visible': visibility}, layout_updates]
            )
        )

    available_ranges = [date_range for date_range in term_ranges.values() if date_range is not None]
    if available_ranges:
        global_start = min(date_range[0] for date_range in available_ranges)
        global_end = max(date_range[1] for date_range in available_ranges)
        fig.update_xaxes(range=term_ranges[default_term] or [global_start, global_end])
        time_range_menu = dict(
            type='dropdown',
            direction='down',
            buttons=build_time_range_buttons(global_start, global_end),
            showactive=True,
            x=0.22,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )
    else:
        time_range_menu = None

    fig.update_yaxes(title_text='Sharpe Spread Z-Score', row=1, col=1)
    fig.update_layout(
        title=f"{ticker_str} {default_term.title()} Sharpe Spread Z-Scores vs Benchmarks ({time_frame_map[default_term]}-Day)",
        height=650,
        template='plotly_dark',
        showlegend=True,
        updatemenus=[
            dict(
                type='dropdown',
                direction='down',
                buttons=timeframe_buttons,
                showactive=True,
                x=0.0,
                xanchor='left',
                y=1.15,
                yanchor='top'
            )
        ] + ([time_range_menu] if time_range_menu is not None else [])
    )
    return fig

def add_detail_asset_trace(fig, asset_sharpe_map, term, visible):
    term_styles = {
        'short': {'color': '#1f77b4', 'dash': 'solid'},
        'mid': {'color': '#ff7f0e', 'dash': 'solid'},
        'long': {'color': '#2ca02c', 'dash': 'solid'}
    }
    style = term_styles[term]
    asset_zscore = calculate_zscore(asset_sharpe_map[term].dropna())
    fig.add_trace(
        go.Scatter(
            x=asset_zscore.index,
            y=asset_zscore,
            mode='lines',
            name=f"{ticker_str} {term.title()} Z-Score",
            legendgroup=f"asset-{term}",
            line=dict(color=style['color'], dash=style['dash']),
            visible=visible
        ),
        row=1,
        col=1
    )

def add_detail_view_traces(fig, symbol, term, benchmark_metric_set, asset_sharpe_map, visible):
    term_styles = {
        'short': {'color': '#1f77b4', 'dash': 'dot'},
        'mid': {'color': '#ff7f0e', 'dash': 'dot'},
        'long': {'color': '#2ca02c', 'dash': 'dot'}
    }
    style = term_styles[term]
    add_detail_asset_trace(fig, asset_sharpe_map, term, visible)
    benchmark_sharpe = calculate_zscore(benchmark_metric_set[term]['sharpe_ratio'].dropna())
    sharpe_spread = calculate_zscore(benchmark_metric_set[term]['sharpe_spread'].dropna())
    relative_spread = calculate_zscore(benchmark_metric_set[term]['spread'].dropna())
    fig.add_trace(
        go.Scatter(
            x=benchmark_sharpe.index,
            y=benchmark_sharpe,
            mode='lines',
            name=f"{symbol} {term.title()} Z-Score",
            legendgroup=f"{symbol}-{term}",
            line=dict(color=style['color'], dash=style['dash']),
            visible=visible
        ),
        row=1,
        col=1
    )
    fig.add_trace(
        go.Scatter(
            x=sharpe_spread.index,
            y=sharpe_spread,
            mode='lines',
            name=f"{symbol} {term.title()} Sharpe Spread Z-Score",
            legendgroup=f"{symbol}-{term}-spread",
            line=dict(color=style['color']),
            visible=visible,
            showlegend=False
        ),
        row=2,
        col=1
    )
    fig.add_trace(
        go.Scatter(
            x=relative_spread.index,
            y=relative_spread,
            mode='lines',
            name=f"{symbol} {term.title()} Relative Spread Z-Score",
            legendgroup=f"{symbol}-{term}-relative",
            line=dict(color=style['color']),
            visible=visible,
            showlegend=False
        ),
        row=3,
        col=1
    )

def build_detail_visibility_mask(dynamic_trace_count, total_traces, view_index, traces_per_view):
    visibility = [False] * dynamic_trace_count + [True] * (total_traces - dynamic_trace_count)
    start = view_index * traces_per_view
    for trace_idx in range(start, min(start + traces_per_view, dynamic_trace_count)):
        visibility[trace_idx] = True
    return visibility

if benchmark_order:
    summary_fig = make_multi_benchmark_summary_figure(spread_plot_data, time_frame_map, length_of_plots, ticker_str)
    summary_fig.show()

    detail_fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=(
            'Risk-Adjusted Return Z-Score Comparison',
            'Sharpe Spread Z-Score',
            'Relative Return Spread Z-Score'
        )
    )

    detail_term_order = ['short', 'mid', 'long']
    default_detail_term = detail_term_order[0]
    detail_view_order = [(symbol, term) for symbol in benchmark_order for term in detail_term_order]
    default_detail_view = (default_benchmark, default_detail_term)
    traces_per_view = 4

    for symbol, term in detail_view_order:
        add_detail_view_traces(
            detail_fig,
            symbol,
            term,
            benchmark_metrics[symbol],
            asset_sharpe_map,
            visible=((symbol, term) == default_detail_view)
        )

    dynamic_trace_count = len(detail_fig.data)
    detail_x_ref = max((trace.x for trace in detail_fig.data if len(trace.x) > 0), key=len, default=None)
    if detail_x_ref is not None:
        for row in (1, 2, 3):
            add_horizontal_zone_trace(detail_fig, row, detail_x_ref, -3, -2, 'rgba(180, 0, 0, 0.30)')
            add_horizontal_zone_trace(detail_fig, row, detail_x_ref, 2, 3, 'rgba(0, 128, 0, 0.30)')
            add_std_reference_lines(detail_fig, row, detail_x_ref)

    total_traces = len(detail_fig.data)
    buttons = []
    for idx, (symbol, term) in enumerate(detail_view_order):
        visibility = build_detail_visibility_mask(dynamic_trace_count, total_traces, idx, traces_per_view)
        buttons.append(
            dict(
                label=f"{symbol} | {term.title()} ({time_frame_map[term]})",
                method='update',
                args=[
                    {'visible': visibility},
                    {'title': f"{ticker_str} vs {symbol} Benchmark Z-Score Detail [{term.title()} {time_frame_map[term]}-Day]"}
                ]
            )
        )

    detail_fig.update_yaxes(title_text='Sharpe Z-Score', row=1, col=1)
    detail_fig.update_yaxes(title_text='Sharpe Spread Z-Score', row=2, col=1)
    detail_fig.update_yaxes(title_text='Relative Spread Z-Score', row=3, col=1)
    detail_start = min((trace.x[0] for trace in detail_fig.data if len(trace.x) > 0), default=None)
    detail_end = max((trace.x[-1] for trace in detail_fig.data if len(trace.x) > 0), default=None)
    if detail_start is not None and detail_end is not None:
        detail_default_start = max(detail_start, detail_end - pd.DateOffset(years=3))
        detail_fig.update_xaxes(range=[detail_default_start, detail_end])
        time_range_menu = dict(
            type='dropdown',
            direction='down',
            buttons=build_time_range_buttons(detail_start, detail_end, axis_count=3),
            showactive=True,
            x=0.18,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )
    else:
        time_range_menu = None
    detail_fig.update_layout(
        title=f"{ticker_str} vs {default_benchmark} Benchmark Z-Score Detail [{default_detail_term.title()} {time_frame_map[default_detail_term]}-Day]",
        height=1200,
        template='plotly_dark',
        updatemenus=[
            dict(
                type='dropdown',
                direction='down',
                buttons=buttons,
                showactive=True,
                x=0.0,
                xanchor='left',
                y=1.15,
                yanchor='top'
            )
        ] + ([time_range_menu] if time_range_menu is not None else [])
    )
    detail_fig.show()
else:
    print('No benchmark data available for benchmark comparison plots.')


In [ ]:
# Block: analyze drawdown, skew, kurtosis, and Gini z-scores

#Return Distribution Analysis: Drawdown, Skew, Kurtosis, Gini Coefficient

# Parameters
dd_window_options = [21, 50, 200]  # Available window options
default_window = dd_window_options[0]
traces_per_window = 6

# Functions
def calculate_max_drawdown(price_series, window=21):
    rolling_max = price_series.rolling(window=window).max()
    drawdown = price_series / rolling_max - 1
    return drawdown.rolling(window=window).min()

def gini_coefficient(array):
    array = np.abs(array)
    if array.size == 0:
        return np.nan
    sorted_array = np.sort(array)
    n = array.size
    cumvals = np.cumsum(sorted_array)
    if cumvals[-1] == 0:
        return 0.0
    return (n + 1 - 2 * np.sum(cumvals) / cumvals[-1]) / n

def zscore(series):
    std = series.std()
    if std == 0 or pd.isna(std):
        return pd.Series(index=series.index, data=np.nan)
    return (series - series.mean()) / std

def calculate_window_metrics(daily_returns, close_series, window):
    max_drawdown_series = calculate_max_drawdown(close_series, window=window).dropna()
    rolling_skew = daily_returns.rolling(window).apply(lambda x: skew(x, bias=False), raw=False).dropna()
    rolling_kurtosis = daily_returns.rolling(window).apply(lambda x: kurtosis(x, fisher=True, bias=False), raw=False).dropna()
    rolling_gini = daily_returns.rolling(window).apply(lambda x: gini_coefficient(x), raw=False).dropna()
    return {
        'max_drawdown': max_drawdown_series,
        'skew_z': zscore(rolling_skew),
        'kurtosis_z': zscore(rolling_kurtosis),
        'gini_z': zscore(rolling_gini)
    }

def add_drawdown_subplot(fig, drawdown_series, window, visible):
    fig.add_trace(
        go.Scatter(
            x=drawdown_series.index,
            y=drawdown_series,
            mode='lines',
            name=f'{window}-Day Max Drawdown',
            line=dict(color='red'),
            fill='tozeroy',
            visible=visible
        ),
        row=1,
        col=1
    )

def add_zscore_subplot(fig, series, window, row, title, color, visible):
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            mode='lines',
            name=f'{window}-Day {title}',
            line=dict(color=color),
            visible=visible
        ),
        row=row,
        col=1
    )
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=[0] * len(series),
            mode='lines',
            line=dict(dash='dash', color='black'),
            showlegend=False,
            visible=visible
        ),
        row=row,
        col=1
    )

def add_skew_subplot(fig, skew_series, window, visible):
    add_zscore_subplot(fig, skew_series, window, 2, 'Skew Z-Score', 'blue', visible)

def add_kurtosis_subplot(fig, kurtosis_series, window, visible):
    add_zscore_subplot(fig, kurtosis_series, window, 3, 'Excess Kurtosis Z-Score', 'purple', visible)

def add_gini_subplot(fig, gini_series, window, visible):
    add_zscore_subplot(fig, gini_series, window, 4, 'Gini Coefficient Z-Score', 'green', visible)

def add_zscore_reference_lines(fig, x_ref, row, levels=(1, 2, 3)):
    for level in levels:
        for value in (level, -level):
            fig.add_trace(
                go.Scatter(
                    x=x_ref,
                    y=[value] * len(x_ref),
                    mode='lines',
                    line=dict(dash='dot', color='gray', width=1),
                    showlegend=False,
                    visible=True
                ),
                row=row,
                col=1
            )

def build_visibility_mask(total_traces, active_window_index, traces_per_window, constant_trace_indices):
    visibility = [False] * total_traces
    start = active_window_index * traces_per_window
    for trace_idx in range(start, start + traces_per_window):
        visibility[trace_idx] = True
    for trace_idx in constant_trace_indices:
        visibility[trace_idx] = True
    return visibility

# Calculate rolling metrics for all windows
close_series = ticker['Close'].dropna()
daily_returns = close_series.pct_change().dropna()
metrics = {window: calculate_window_metrics(daily_returns, close_series, window) for window in dd_window_options}

# Create subplots
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        'Rolling Max Drawdown',
        'Rolling Skew Z-Score',
        'Rolling Excess Kurtosis Z-Score',
        'Rolling Gini Coefficient Z-Score'
    )
)

# Add traces for each window
for window in dd_window_options:
    visible = window == default_window
    window_metrics = metrics[window]
    add_drawdown_subplot(fig, window_metrics['max_drawdown'], window, visible)
    add_skew_subplot(fig, window_metrics['skew_z'], window, visible)
    add_kurtosis_subplot(fig, window_metrics['kurtosis_z'], window, visible)
    add_gini_subplot(fig, window_metrics['gini_z'], window, visible)

# Add horizontal reference lines for z-score panels
x_ref = metrics[default_window]['skew_z'].index
for row in (2, 3, 4):
    add_zscore_reference_lines(fig, x_ref, row)

# Prepare dropdown visibility settings
total_traces = len(fig.data)
first_constant_trace = traces_per_window * len(dd_window_options)
constant_trace_indices = list(range(first_constant_trace, total_traces))

buttons = []
for idx, window in enumerate(dd_window_options):
    visibility = build_visibility_mask(total_traces, idx, traces_per_window, constant_trace_indices)
    buttons.append(
        dict(
            label=str(window),
            method='update',
            args=[
                {'visible': visibility},
                {'title': f'{ticker_str} Rolling Risk Metrics Z-Scores ({window}-Day Window)'}
            ],
        )
    )

# Update y-axes
fig.update_yaxes(title_text='Drawdown', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Skew Z', row=2, col=1)
fig.update_yaxes(title_text='Excess Kurtosis Z', row=3, col=1)
fig.update_yaxes(title_text='Gini Z', row=4, col=1)

# Add dropdown menu
fig.update_layout(
    updatemenus=[
        dict(
            type='dropdown',
            direction='down',
            buttons=buttons,
            showactive=True,
            x=0.1,
            xanchor='left',
            y=1.1,
            yanchor='top'
        )
    ],
    title=f'{ticker_str} Rolling Risk Metrics Z-Scores ({default_window}-Day Window)',
    height=1500,
    template='plotly_white',
    showlegend=False
)

# Show figure
fig.show()
